
# Notebook 08 — Compare three agents: does curation actually help?

The proof. We run the **same four hard questions** against the three agents built earlier — all on the same data — and compare accuracy:

| Agent | Built in | What it has |
|---|---|---|
| **Baseline** | 03 | Tables only, minimal instructions |
| **Metric View** | 03b | One governed metric view (pinned joins + KPI formulas) |
| **Knowledge Store** | 04 | Measures, filters, fields, joins, synonyms, example SQL |

**What to expect** — and why it's honest:
- **Baseline** struggles: it has to guess joins and formulas.
- **Knowledge Store** does best across the board: the curation covers all four patterns.
- **Metric View** nails the KPIs it encodes (scrap rate) but **can't answer event-level questions** (defect-rate-by-line, best-shift) — those columns aren't in the view. That's the trade-off: deterministic within its perimeter, limited outside it.

## The four questions

| # | Question | Why it's hard |
|---|---|---|
| Q1 | Scrap rate, Michigan plants, last 5 days of 2024 | state join + date filter + ratio |
| Q2 | Distinct plants with a Maintenance-status line | static status filter + join |
| Q3 | Lines with >5% defect rate in 2024 | CASE/HAVING across production_events |
| Q4 | Best-quality shift in Dec 2024 | events→operators join + MIN |

**Before you start:** run **03** (baseline), **03b** (metric view), **04** (knowledge store).

**Compute:** Serverless.

## Load config and connect to the three agents

In [ ]:
%run ./00_workshop_config

In [ ]:
import re
import time
import json
import requests
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"


_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
BASELINE_ID = _cfg.get(CFG_KEY_BASELINE, "")
CURATED_ID = _cfg.get(CFG_KEY_CURATED, "")
METRIC_ID = _cfg.get(CFG_KEY_METRIC_VIEW, "")

if not CURATED_ID:
    raise RuntimeError("genie_space_id (Knowledge Store agent) not found. Run notebook 04 first.")

# Agents to compare, in narrative order. Skip any that weren't created.
AGENTS = [(name, sid) for name, sid in [
    ("Baseline", BASELINE_ID),
    ("Metric View", METRIC_ID),
    ("Knowledge Store", CURATED_ID),
] if sid]

print("Comparing:")
for name, sid in AGENTS:
    print(f"  {name:<16} {genie_ui_room_url(sid)}")
if len(AGENTS) < 3:
    print("\nNote: run 03 / 03b / 04 to compare all three. Continuing with what exists.")

## Scoring function

Sends a question to a Genie agent via the Conversation API, waits for the answer, extracts the first number, and compares it to ground-truth SQL within a tolerance.

In [ ]:
def _extract_number(text):
    if text is None:
        return None
    nums = re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    return float(nums[0]) if nums else None


def run_benchmarks(benchmarks, space_id, label):
    """Ask each question, score against ground truth. Returns (results, pass_rate)."""
    print(f"\n{label}")
    results = []
    passes = 0
    for i, b in enumerate(benchmarks, 1):
        gt_val = float(spark.sql(b["gt"]).collect()[0][0])
        genie_val, status = None, "FAIL"
        try:
            start = requests.post(
                f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
                headers=headers, json={"content": b["q"]},
            )
            if start.status_code != 200:
                print(f"  Q{i}: SKIP (start-conversation {start.status_code})")
                results.append((i, b["q"], gt_val, None, "ERROR"))
                continue
            d = start.json()
            cid, mid = d.get("conversation_id"), d.get("message_id")
            for _ in range(40):
                time.sleep(4)
                poll = requests.get(
                    f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}",
                    headers=headers,
                )
                if poll.status_code != 200:
                    continue
                msg = poll.json()
                st = msg.get("status", "")
                if st == "COMPLETED":
                    for att in msg.get("attachments", []):
                        aid = att.get("attachment_id") or att.get("id")
                        if not att.get("query") or not aid:
                            continue
                        qr = requests.get(
                            f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result/{aid}",
                            headers=headers,
                        )
                        if qr.status_code == 200:
                            arr = qr.json().get("statement_response", {}).get("result", {}).get("data_array", [])
                            if arr and arr[0]:
                                genie_val = _extract_number(arr[0][0])
                    if genie_val is not None and gt_val != 0:
                        status = "PASS" if abs(genie_val - gt_val) / abs(gt_val) * 100 <= BENCHMARK_TOLERANCE_PCT else "FAIL"
                    elif genie_val is not None and gt_val == 0:
                        status = "PASS" if genie_val == 0 else "FAIL"
                    break
                if st in ("FAILED", "CANCELLED"):
                    break
        except Exception as e:
            print(f"  Q{i}: ERROR ({str(e)[:100]})")
        if status == "PASS":
            passes += 1
        print(f"  Q{i}: {status} (GT={gt_val}, Genie={genie_val})")
        results.append((i, b["q"], gt_val, genie_val, status))
    rate = (passes / len(benchmarks) * 100) if benchmarks else 0
    print(f"  Pass rate: {rate:.0f}% ({passes}/{len(benchmarks)})")
    return results, rate


print("Scoring function ready.")

## The four hard questions

In [ ]:
hard_questions = [
    {"q": "What is the scrap rate for Michigan plants for the last 5 days of 2024? Return the total scrap_count divided by total units_produced as a percentage from quality_metrics_daily.",
     "gt": f"SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND CAST(q.date AS DATE) >= DATE '2024-12-27' AND CAST(q.date AS DATE) <= DATE '2024-12-31'",
     "short": "Scrap rate Michigan last 5 days"},
    {"q": "How many distinct plants have at least one production line with status Maintenance? Return the count of distinct plants.",
     "gt": f"SELECT CAST(COUNT(DISTINCT p.plant_id) AS BIGINT) FROM {fqn}.production_lines pl JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE pl.status = 'Maintenance'",
     "short": "Plants with Maintenance lines"},
    {"q": "How many production lines have a high historical defect rate — meaning more than 5% of their production_events in 2024 were defect_detected out of total unit_produced events?",
     "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id, COUNT(CASE WHEN event_type = 'defect_detected' THEN 1 END) AS defects, COUNT(CASE WHEN event_type = 'unit_produced' THEN 1 END) AS produced FROM {fqn}.production_events WHERE YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING produced > 0 AND (100.0 * defects / produced) > 5.0 ) t",
     "short": "Lines with >5% defect rate"},
    {"q": "How many defect_detected events did the best-quality shift have in December 2024? Best quality = fewest defects. Join production_events to operators by operator_id to get the shift, group by shift, and return only the lowest defect count.",
     "gt": f"SELECT CAST(MIN(defect_count) AS BIGINT) FROM ( SELECT o.shift, COUNT(*) AS defect_count FROM {fqn}.production_events pe JOIN {fqn}.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type = 'defect_detected' AND CAST(pe.event_date AS DATE) >= DATE '2024-12-01' AND CAST(pe.event_date AS DATE) <= DATE '2024-12-31' GROUP BY o.shift ) t",
     "short": "Best-quality shift Dec 2024"},
]
print(f"{len(hard_questions)} hard questions ready.")

## Run the comparison

Each question goes to every agent. This calls the live Genie API, so it takes a few minutes.

In [ ]:
all_results = {}   # name -> (results, rate)
for name, sid in AGENTS:
    all_results[name] = run_benchmarks(hard_questions, sid, f"=== {name} ===")

## Results side by side

In [ ]:
names = [n for n, _ in AGENTS]
header = f"{'Q#':<4} {'Question':<34}" + "".join(f"{n:<18}" for n in names)
print("=" * len(header))
print("THREE-WAY COMPARISON")
print("=" * len(header))
print(header)
print("-" * len(header))
for idx, hq in enumerate(hard_questions):
    row = f"Q{idx+1:<3} {hq['short']:<34}"
    for n in names:
        res = all_results[n][0][idx]
        row += f"{res[4] + ' (' + str(res[3]) + ')':<18}"
    print(row)
print("-" * len(header))
rate_row = f"{'Pass rate':<39}"
for n in names:
    rate_row += f"{str(int(all_results[n][1])) + '%':<18}"
print(rate_row)
print()
print("Reading the result:")
if "Knowledge Store" in all_results and "Baseline" in all_results:
    ks, base = all_results["Knowledge Store"][1], all_results["Baseline"][1]
    if ks > base:
        print(f"  - Knowledge Store beat Baseline ({int(ks)}% vs {int(base)}%): curated joins,")
        print(f"    measures, and example SQL are doing the work.")
    else:
        print(f"  - Knowledge Store {int(ks)}% vs Baseline {int(base)}%: on clean synthetic data")
        print(f"    Genie can sometimes guess correctly — curation still ensures reliability.")
if "Metric View" in all_results:
    print("  - Metric View is deterministic on the KPIs it encodes (e.g. scrap rate) but")
    print("    can't answer event-level questions (Q3/Q4) — those columns aren't in the view.")
    print("    That's the perimeter trade-off: pair metric views with tables for open exploration.")

## Save run history

Appends the results to `genie_benchmark_runs` so notebook **11** (monitoring) can chart pass-rate trends.

In [ ]:
ts = datetime.now().isoformat()
sch = StructType([
    StructField("benchmark_id", IntegerType()),
    StructField("question", StringType()),
    StructField("ground_truth", DoubleType()),
    StructField("genie_answer", DoubleType()),
    StructField("status", StringType()),
    StructField("run_timestamp", StringType()),
    StructField("pass_rate", DoubleType()),
    StructField("run_label", StringType()),
])
rows = []
for name, (results, rate) in all_results.items():
    label = "compare_" + name.lower().replace(" ", "_")
    rows += [(r[0], r[1], r[2], r[3], r[4], ts, rate, label) for r in results]

tbl = f"{fqn}.genie_benchmark_runs"
try:
    spark.createDataFrame(rows, sch).write.mode("append").saveAsTable(tbl)
except Exception:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")
    spark.createDataFrame(rows, sch).write.saveAsTable(tbl)
print(f"Saved {len(rows)} results to {tbl}")

## Validate in the UI (most accurate)

The programmatic scorer compares one number with a tolerance. The Genie **Benchmark** tab compares full result sets and is the most accurate evaluator.

1. Open the **Knowledge Store** agent link above.
2. Click the **Benchmark** tab → **Start new run**.
3. Review failures; for each, **Review proposed fixes** and accept the knowledge snippets that match your conventions, or **Update ground truth** if Genie's SQL is equivalent.
4. Re-run until green. Target **80%+ before UAT**, higher on clean data.

**Next:** Notebook 09 — Security and governance (column masking).